 Derived from
 
 https://www.axonlab.org/hcph-sops/data-management/edf-to-bids/
 
 https://github.com/TheAxonLab/hcph-sops/blob/mkdocs/code/eyetracking/convert.py

Make sure the python version >=3.7 to support the statement

In [1]:
from __future__ import annotations 
from pathlib import Path
import pandas as pd
import numpy as np
from pyedfread import read_edf
from collections import defaultdict
from itertools import product, groupby
from warnings import warn
import re

In [2]:
# Global variable from 
# https://github.com/TheAxonLab/hcph-sops/blob/mkdocs/code/eyetracking/convert.py

# If setting WRITE_RAW_EDF as True, no preprocessing will be conducted on the edf data
WRITE_RAW_EDF = True
# -------------------------------------------------------------------------------
DEFAULT_EYE = "right"
DEFAULT_FREQUENCY = 1000 #It is 1000
DEFAULT_MODE = "P-CR"
DEFAULT_SCREEN = (0, 800, 0, 600)

# EyeLink calibration coordinates from
# https://www.sr-research.com/calibration-coordinate-calculator/
# Affect the performance?
EYELINK_CALIBRATION_COORDINATES = [
    (400, 300),
    (400, 51),
    (400, 549),
    (48, 300),
    (752, 300),
    (48, 51),
    (752, 51),
    (48, 549),
    (752, 549),
    (224, 176),
    (576, 176),
    (224, 424),
    (576, 424),
]

EYE_CODE_MAP = defaultdict(lambda: "unknown", {"R": "right", "L": "left", "RL": "both"})
EDF2BIDS_COLUMNS = {
    "g": '',
    "p": "pupil",
    "h": "href",
    "r": "raw",
    "fg": "fast",
    "fh": "fast_href",
    "fr": "fast_raw",
}

BIDS_COLUMNS_ORDER = (
    [f"eye{num}_{c}_coordinate" for num, c in product((1, 2), ("x", "y"))]
    + [f"eye{num}_pupil_size" for num in (1, 2)]
    + [f"eye{num}_pupil_{c}_coordinate" for num, c in product((1, 2), ("x", "y"))]
    + [f"eye{num}_fixation" for num in (1, 2)]
    + [f"eye{num}_saccade" for num in (1, 2)]
    + [f"eye{num}_blink" for num in (1, 2)]
    + [f"eye{num}_href_{c}_coordinate" for num, c in product((1, 2), ("x", "y"))]
    + [f"eye{num}_{c}_velocity" for num, c in product((1, 2), ("x", "y"))]
    + [f"eye{num}_href_{c}_velocity" for num, c in product((1, 2), ("x", "y"))]
    + [f"eye{num}_raw_{c}_velocity" for num, c in product((1, 2), ("x", "y"))]
    + [f"fast_{c}_velocity" for c in ("x", "y")]
    + [f"fast_{kind}_{c}_velocity" for kind, c in product(("href", "raw"), ("x", "y"))]
    + [f"screen_ppdeg_{c}_coordinate" for c in ("x", "y")]
    + ["timestamp"]
)


Read in the edf file

In [3]:
# scanner -> ET
subject_idx = 1 

# ET -> scanner
# subject_idx = 2


T_idx = 1

DATA_PATH = Path("/Users/cag/Documents/Dataset/datasets/250423/edf_250423/")

if subject_idx == 1:    
    edf_name = f"034455_fixation_dots_T1weighted_2025-04-24_16h12.45.920.EDF"    
elif subject_idx == 2:
    edf_name = f"..." 
elif subject_idx == 3: 
    edf_name = f"..."
    # "OT4.EDF"
else:
    edf_name = f"..."
    

file_path = str(DATA_PATH / edf_name)
print(file_path)
ori_recording, ori_events, ori_messages = read_edf(file_path)
# The first timestamp of  `recording`
# print(f" {ori_recording[100000:100100]}")
# print(ori_messages)
# print(ori_events)
# print(messages)
ori_messages = ori_messages.rename(
    columns={
        # Normalize weird header names generated by pyedfread
        "message": "trialid",
        "trial": "trial",
        # Convert some BIDS columns
        "time": "timestamp",
    }
)

recording = ori_recording
messages = ori_messages
events = ori_events
print(len(messages))
print(f'\nThe entire info of `message`: \n{messages[:]}')
# recording.columns

/Users/cag/Documents/Dataset/datasets/250423/edf_250423/034455_fixation_dots_T1weighted_2025-04-24_16h12.45.920.EDF
50

The entire info of `message`: 
    timestamp  trial                                            trialid
0      624794     -1  !CAL \n>>>>>>> CALIBRATION (HV5,P-CR) FOR RIGH...
1      624794     -1                           !CAL Calibration points:
2      624794     -1                !CAL -36.8, -43.5         0,      0
3      624794     -1                !CAL -35.3, -59.8         0,  -2457
4      624794     -1                !CAL -36.9, -27.8         0,   2457
5      624794     -1                !CAL -60.2, -45.3     -3474,      0
6      624794     -1                 !CAL -7.3, -42.1      3474,      0
7      624794     -1  !CAL eye check box: (L,R,T,B)\n\t  -66    -2  ...
8      624794     -1  !CAL href cal range: (L,R,T,B)\n\t-5211  5211 ...
9      624794     -1  !CAL Cal coeff:(X=a+bx+cy+dxx+eyy,Y=f+gx+goaly...
10     624794     -1      !CAL Prenormalize: offx, offy =

# 1 Parsing the messages

In [7]:
messages = messages.rename(
    columns={c: c.strip() for c in messages.columns.values}
).drop_duplicates()

In [8]:

# Extract calibration headers
_cal_hdr = ori_messages.trialid.str.startswith("!CAL")
calibration = ori_messages[_cal_hdr]
# messages = messages.drop(messages.index[_cal_hdr])
print(calibration)

    timestamp  trial                                            trialid
0      624794     -1  !CAL \n>>>>>>> CALIBRATION (HV5,P-CR) FOR RIGH...
1      624794     -1                           !CAL Calibration points:
2      624794     -1                !CAL -36.8, -43.5         0,      0
3      624794     -1                !CAL -35.3, -59.8         0,  -2457
4      624794     -1                !CAL -36.9, -27.8         0,   2457
5      624794     -1                !CAL -60.2, -45.3     -3474,      0
6      624794     -1                 !CAL -7.3, -42.1      3474,      0
7      624794     -1  !CAL eye check box: (L,R,T,B)\n\t  -66    -2  ...
8      624794     -1  !CAL href cal range: (L,R,T,B)\n\t-5211  5211 ...
9      624794     -1  !CAL Cal coeff:(X=a+bx+cy+dxx+eyy,Y=f+gx+goaly...
10     624794     -1      !CAL Prenormalize: offx, offy = -36.815 -43.5
11     624794     -1         !CAL Gains: cx:88.187 lx:124.778 rx:51.882
12     624794     -1       !CAL Gains: cy:171.577 ty:160.046 by:

In [9]:
# Extracting the StartTime and StopTime metadata.
message_first_trigger = '!MODE RECORD CR 1000 2 0 R'
message_last_trigger = 'ET: eye-tracker stopped'
metadata = {
    'StopTime': None,
    'StartTime': None
}

# Find Start time
start_rows = messages.trialid.str.contains(
    message_first_trigger, case=False, regex=True
)
stop_rows = messages.trialid.str.contains(
    message_last_trigger, case=False, regex=True
)


# Extract calibration headers
_cal_hdr = messages.trialid.str.startswith("!CAL")
calibration = messages[_cal_hdr]
messages = messages.drop(messages.index[_cal_hdr])

# Pick the LAST of the start messages
metadata["StartTime"] = (
    int(messages[start_rows].timestamp.values[-1])
    if start_rows.any()
    else None
)

# Pick the FIRST of the stop messages
metadata["StopTime"] = (
    int(messages[stop_rows].timestamp.values[0])
    if stop_rows.any()
    else None
)

# Drop start and stop messages from messages dataframe
messages = messages.loc[~start_rows & ~stop_rows, :]

metadata

/var/folders/x4/yl1kbpks5sxc3345y3ttk6gm0000gn/T/ipykernel_94087/906092332.py:25: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  int(messages[start_rows].timestamp.values[-1])


{'StopTime': None, 'StartTime': 910995}

In [10]:
# Extracting basic metadata.
# !MODE RECORD CR 1000 2 0 R

mode_record = messages.trialid.str.startswith("!MODE RECORD")

meta_record = {
    "freq": DEFAULT_FREQUENCY,
    "mode": DEFAULT_MODE,
    "eye": DEFAULT_EYE,
}

if mode_record.any():
    try:
        meta_record = re.match(
            r"\!MODE RECORD (?P<mode>\w+) (?P<freq>\d+) \d \d (?P<eye>[RL]+)",
            messages[mode_record].trialid.iloc[-1].strip(),
        ).groupdict()

        meta_record["eye"] = EYE_CODE_MAP[meta_record["eye"]]
        meta_record["mode"] = (
            "P-CR" if meta_record["mode"] == "CR" else meta_record["mode"]
        )
    except AttributeError:
        warn(
            "Error extracting !MODE RECORD message, "
            "using default frequency, mode, and eye"
        )
    finally:
        messages = messages.loc[~mode_record]

eye = (
    ("right", "left") if meta_record["eye"] == "both" else (meta_record["eye"],)
)

metadata["SamplingFrequency"] = int(meta_record["freq"])
metadata["EyeTrackingMethod"] = meta_record["mode"]
metadata["RecordedEye"] = meta_record["eye"]

In [11]:
# Extracting screen parameters.
# GAZE_COORDS 0.00 0.00 800.00 600.00

# Extract GAZE_COORDS message signaling start of recording
gaze_msg = messages.trialid.str.startswith("GAZE_COORDS")

metadata["ScreenAOIDefinition"] = [
    "square",
    DEFAULT_SCREEN,
]
if gaze_msg.any():
    try:
        gaze_record = re.match(
            r"GAZE_COORDS (\d+\.\d+) (\d+\.\d+) (\d+\.\d+) (\d+\.\d+)",
            messages[gaze_msg].trialid.iloc[-1].strip(),
        ).groups()
        metadata["ScreenAOIDefinition"][1] = [
            int(round(float(gaze_record[0]))),
            int(round(float(gaze_record[2]))),
            int(round(float(gaze_record[1]))),
            int(round(float(gaze_record[3]))),
        ]
    except AttributeError:
        warn("Error extracting GAZE_COORDS")
    finally:
        messages = messages.loc[~gaze_msg]
        
print(metadata)

{'StopTime': None, 'StartTime': 910995, 'SamplingFrequency': 1000, 'EyeTrackingMethod': 'P-CR', 'RecordedEye': 'right', 'ScreenAOIDefinition': ['square', [0, 800, 0, 600]]}


In [12]:
# Extracting parameters of the pupil fit model.
# ELCL_PROC ELLIPSE (5)
# ELCL_EFIT_PARAMS 1.01 4.00  0.15 0.05  0.65 0.65  0.00 0.00 0.30
# Extract ELCL_PROC AND ELCL_EFIT_PARAMS to extract pupil fit method
pupilfit_msg = messages.trialid.str.startswith("ELCL_PROC")

if pupilfit_msg.any():
    try:
        pupilfit_method = [
            val
            for val in messages[pupilfit_msg]
            .trialid.iloc[-1]
            .strip()
            .split(" ")[1:]
            if val
        ]
        metadata["PupilFitMethod"] = pupilfit_method[0].lower()
        metadata["PupilFitMethodNumberOfParameters"] = int(
            pupilfit_method[1].strip("(").strip(")")
        )
    except AttributeError:
        warn("Error extracting ELCL_PROC (pupil fitting method)")
    finally:
        messages = messages.loc[~pupilfit_msg]

pupilfit_msg_params = messages.trialid.str.startswith("ELCL_EFIT_PARAMS")
if pupilfit_msg_params.any():
    rows = messages[pupilfit_msg_params]
    row = rows.trialid.values[-1].strip().split(" ")[1:]
    try:
        metadata["PupilFitParameters"] = [
            tuple(float(val) for val in vals)
            for k, vals in groupby(row, key=bool)
            if k
        ]
    except AttributeError:
        warn("Error extracting ELCL_EFIT_PARAMS (pupil fitting parameters)")
    finally:
        messages = messages.loc[~pupilfit_msg_params]
        
metadata

{'StopTime': None,
 'StartTime': 910995,
 'SamplingFrequency': 1000,
 'EyeTrackingMethod': 'P-CR',
 'RecordedEye': 'right',
 'ScreenAOIDefinition': ['square', [0, 800, 0, 600]],
 'PupilFitMethod': 'ellipse',
 'PupilFitMethodNumberOfParameters': 5,
 'PupilFitParameters': [(1.01, 4.0),
  (0.15, 0.05),
  (0.65, 0.65),
  (0.0, 0.0, 0.3)]}

In [13]:
# Calibration validation.
# VALIDATE R 4POINT 4 RIGHT at 752,300 OFFSET 0.35 deg. -8.7,-3.8 pix.
# Extract VALIDATE messages for a calibration validation
validation_msg = messages.trialid.str.startswith("VALIDATE")

if validation_msg.any():
    metadata["ValidationPosition"] = []
    metadata["ValidationErrors"] = []

for i_row, validate_row in enumerate(messages[validation_msg].trialid.values):
    prefix, suffix = validate_row.split("OFFSET")
    validation_eye = (
        f"eye{eye.index('right') + 1}"
        if "RIGHT" in prefix
        else f"eye{eye.index('left') + 1}"
    )
    validation_coords = [
        int(val.strip())
        for val in prefix.rsplit("at", 1)[-1].split(",")
        if val.strip()
    ]
    metadata["ValidationPosition"].append(
        [validation_eye, validation_coords]
    )

    validate_values = [
        float(val)
        for val in re.match(
            r"(-?\d+\.\d+) deg\.\s+(-?\d+\.\d+),(-?\d+\.\d+) pix\.",
            suffix.strip(),
        ).groups()
    ]

    metadata["ValidationErrors"].append(
        (validation_eye, validate_values[0], tuple(validate_values[1:]))
    )
messages = messages.loc[~validation_msg]

print(messages)
print(metadata)

    timestamp  trial                                            trialid
33     703078     -1  NO Reply is disabled for function eyelink_cal_...
34     910988     -1                             Key s trigger response
35     910989     -1                               Hello tracker record
36     910994     -1                               RECCFG CR 1000 2 0 R
37     910994     -1                                      ELCLCFG TOWER
39     910994     -1                                THRESHOLDS R 84 208
43     911491     -1                                 Bye tracker record
44     911494     -1                            T1w_LIBRE stimuli start
45     911494     -1                         Stimuli after 0 seconds...
46    1011486     -1                       Stimuli after 100 seconds...
47    1111488     -1                       Stimuli after 200 seconds...
48    1211490     -1                       Stimuli after 300 seconds...
49    1311508     -1                       Stimuli after 400 sec

In [14]:
# Extracting final bits of metadata.
# Extract THRESHOLDS messages prior recording and process last
thresholds_msg = messages.trialid.str.startswith("THRESHOLDS")
if thresholds_msg.any():
    metadata["PupilThreshold"] = [None] * len(eye)
    metadata["CornealReflectionThreshold"] = [None] * len(eye)
    thresholds_chunks = (
        messages[thresholds_msg].trialid.iloc[-1].strip().split(" ")[1:]
    )
    eye_index = eye.index(EYE_CODE_MAP[thresholds_chunks[0]])
    metadata["PupilThreshold"][eye_index] = int(thresholds_chunks[-2])
    metadata["CornealReflectionThreshold"][eye_index] = int(
        thresholds_chunks[-1]
    )
messages = messages.loc[~thresholds_msg]
print(messages)
print(metadata)

    timestamp  trial                                            trialid
33     703078     -1  NO Reply is disabled for function eyelink_cal_...
34     910988     -1                             Key s trigger response
35     910989     -1                               Hello tracker record
36     910994     -1                               RECCFG CR 1000 2 0 R
37     910994     -1                                      ELCLCFG TOWER
43     911491     -1                                 Bye tracker record
44     911494     -1                            T1w_LIBRE stimuli start
45     911494     -1                         Stimuli after 0 seconds...
46    1011486     -1                       Stimuli after 100 seconds...
47    1111488     -1                       Stimuli after 200 seconds...
48    1211490     -1                       Stimuli after 300 seconds...
49    1311508     -1                       Stimuli after 400 seconds...
{'StopTime': None, 'StartTime': 910995, 'SamplingFrequency': 100

In [15]:
# Flush the remaining messages as a metadata entry.
# Consume the remainder of messages

if not messages.empty:
    metadata["LoggedMessages"] = [
        (int(msg_timestamp), msg.strip())
        for msg_timestamp, msg in messages[["timestamp", "trialid"]].values
    ]
    
print(messages)
print(metadata)

    timestamp  trial                                            trialid
33     703078     -1  NO Reply is disabled for function eyelink_cal_...
34     910988     -1                             Key s trigger response
35     910989     -1                               Hello tracker record
36     910994     -1                               RECCFG CR 1000 2 0 R
37     910994     -1                                      ELCLCFG TOWER
43     911491     -1                                 Bye tracker record
44     911494     -1                            T1w_LIBRE stimuli start
45     911494     -1                         Stimuli after 0 seconds...
46    1011486     -1                       Stimuli after 100 seconds...
47    1111488     -1                       Stimuli after 200 seconds...
48    1211490     -1                       Stimuli after 300 seconds...
49    1311508     -1                       Stimuli after 400 seconds...
{'StopTime': None, 'StartTime': 910995, 'SamplingFrequency': 100

# 2 Parsing the recording dataframe

In [16]:
recording = ori_recording
ori_recording

,time,px_left,px_right,py_left,py_right,hx_left,hx_right,hy_left,hy_right,pa_left,...,fgyvel,fhxvel,fhyvel,frxvel,fryvel,flags,input,buttons,htype,errors
0,910995.0,-32768.0,-4219.0,-32768.0,-5121.0,-32768.0,420.0,-32768.0,360.0,-32768.0,...,1.401298e-45,6.168592e+22,0.0,0.0,0.0,32641.0,32768.0,0.0,-32768.0,0.0
1,910996.0,-32768.0,-4222.0,-32768.0,-5118.0,-32768.0,416.0,-32768.0,363.0,-32768.0,...,1.401298e-45,6.168592e+22,0.0,0.0,0.0,24449.0,32768.0,0.0,-32768.0,0.0
2,910997.0,-32768.0,-4226.0,-32768.0,-5116.0,-32768.0,412.0,-32768.0,366.0,-32768.0,...,1.401298e-45,6.168592e+22,0.0,0.0,0.0,24449.0,32768.0,0.0,-32768.0,0.0
3,910998.0,-32768.0,-4223.0,-32768.0,-5124.0,-32768.0,416.0,-32768.0,356.0,-32768.0,...,1.401298e-45,6.168592e+22,0.0,0.0,0.0,24449.0,32768.0,0.0,-32768.0,0.0
4,910999.0,-32768.0,-4215.0,-32768.0,-5132.0,-32768.0,424.0,-32768.0,345.0,-32768.0,...,1.401298e-45,6.168592e+22,0.0,0.0,0.0,24449.0,32768.0,0.0,-32768.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
428164,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000e+00,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
428165,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000e+00,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
428166,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000e+00,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
428167,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000e+00,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [17]:
# Curation of the input dataframe
# Normalize timestamps (should be int and strictly positive)
recording = recording.astype({"time": int})
recording = recording[recording["time"] > 0]
raw_recording_len = len(recording)
print(f'raw_recording length: {raw_recording_len}')

recording = recording.rename(
    columns={
#         # Fix buggy header names generated by pyedfread
#         "fhxyvel": "fhxvel",
#         "frxyvel": "frxvel",
        # Normalize weird header names generated by pyedfread
        "rx": "screen_ppdeg_x_coordinate",
        "ry": "screen_ppdeg_y_coordinate",
        # Convert some BIDS columns
        "time": "timestamp",
    }
)

# Split extra columns from the dataframe
extra = recording[["flags", "input", "htype"]]
recording = recording.drop(columns=["flags", "input", "htype"])
print(len(recording))

# Remove columns that are always very close to zero
recording = recording.loc[:, (recording.abs() > 1e-8).any(axis=0)]
# Remove columns that are always 1e8 or more
recording = recording.loc[:, (recording.abs() < 1e8).any(axis=0)]
# Replace unreasonably high values with NaNs
recording = recording.replace({1e8: np.nan})

assert len(recording) == raw_recording_len

raw_recording length: 425457
425457


In [18]:
# Remove columns that do not apply (e.g., only one eye recorded).
# Drop one eye's columns if not interested in "both"
print(f'The eye we take care of {eye}')
remove_eye = set(("left", "right")) - set(eye)
if remove_eye:
    remove_eye = remove_eye.pop()  # Drop set decoration
    recording = recording.reindex(
        columns=[c for c in recording.columns if remove_eye not in c]
    )
    
columns = recording.columns
print("Columns:")
print(columns)
recording

The eye we take care of ('right',)
Columns:
Index(['timestamp', 'px_right', 'py_right', 'hx_right', 'hy_right', 'pa_right',
       'gx_right', 'gy_right', 'screen_ppdeg_x_coordinate',
       'screen_ppdeg_y_coordinate', 'fhxvel', 'frxvel'],
      dtype='object')


,timestamp,px_right,py_right,hx_right,hy_right,pa_right,gx_right,gy_right,screen_ppdeg_x_coordinate,screen_ppdeg_y_coordinate,fhxvel,frxvel
0,910995,-4219.0,-5121.0,420.0,360.0,847.0,442.600006,336.399994,26.5,26.5,6.168592e+22,0.000000e+00
1,910996,-4222.0,-5118.0,416.0,363.0,847.0,442.200012,336.799988,26.5,26.5,6.168592e+22,0.000000e+00
2,910997,-4226.0,-5116.0,412.0,366.0,847.0,441.799988,337.100006,26.5,26.5,6.168592e+22,0.000000e+00
3,910998,-4223.0,-5124.0,416.0,356.0,847.0,442.100006,336.000000,26.5,26.5,6.168592e+22,0.000000e+00
4,910999,-4215.0,-5132.0,424.0,345.0,847.0,443.000000,335.000000,26.5,26.5,6.168592e+22,0.000000e+00
...,...,...,...,...,...,...,...,...,...,...,...,...
425452,1336447,-4641.0,-5483.0,-45.0,-53.0,879.0,395.299988,294.600006,26.5,26.5,6.168592e+22,1.958460e+26
425453,1336448,-4639.0,-5483.0,-43.0,-53.0,880.0,395.600006,294.500000,26.5,26.5,6.168592e+22,1.958460e+26
425454,1336449,-4637.0,-5492.0,-41.0,-65.0,879.0,395.700012,293.299988,26.5,26.5,6.168592e+22,1.958460e+26
425455,1336450,-4636.0,-5505.0,-41.0,-80.0,877.0,395.799988,291.700012,26.5,26.5,6.168592e+22,1.958460e+26


In [19]:
# Clean-up pupil size and gaze position. 
# These are the parameters we most likely we care for, so special curation is applied:
screen_resolution = [800, 600]

for eyenum, eyename in enumerate(eye):
    # Clean-up implausible values for pupil area (pa)
    recording.loc[
        recording[f"pa_{eyename}"] < 1, f"pa_{eyename}"
    ] = np.nan
    recording = recording.rename(
        columns={f"pa_{eyename}": f"eye{eyenum + 1}_pupil_size"}
    )
    print(f"pa_{eyename} renamed as: eye{eyenum + 1}_pupil_size")
    # Clean-up implausible values for gaze x position
    recording.loc[
        (recording[f"gx_{eyename}"] < 0)
        | (recording[f"gx_{eyename}"] > screen_resolution[0]),
        f"gx_{eyename}",
    ] = np.nan
    # Clean-up implausible values for gaze y position
    recording.loc[
        (recording[f"gy_{eyename}"] <= 0)
        | (recording[f"gy_{eyename}"] > screen_resolution[1]),
        f"gy_{eyename}",
    ] = np.nan
    
print(recording)
assert len(recording) == raw_recording_len

pa_right renamed as: eye1_pupil_size
        timestamp  px_right  py_right  hx_right  hy_right  eye1_pupil_size  \
0          910995   -4219.0   -5121.0     420.0     360.0            847.0   
1          910996   -4222.0   -5118.0     416.0     363.0            847.0   
2          910997   -4226.0   -5116.0     412.0     366.0            847.0   
3          910998   -4223.0   -5124.0     416.0     356.0            847.0   
4          910999   -4215.0   -5132.0     424.0     345.0            847.0   
...           ...       ...       ...       ...       ...              ...   
425452    1336447   -4641.0   -5483.0     -45.0     -53.0            879.0   
425453    1336448   -4639.0   -5483.0     -43.0     -53.0            880.0   
425454    1336449   -4637.0   -5492.0     -41.0     -65.0            879.0   
425455    1336450   -4636.0   -5505.0     -41.0     -80.0            877.0   
425456    1336451   -4636.0   -5518.0     -42.0     -96.0            876.0   

          gx_right    gy_r

In [20]:
# Munging columns to comply with BIDS. 
# At this point, the dataframe is almost ready for writing out as BIDS.
# Interpolate BIDS column names
columns = list(
    set(recording.columns)
    - set(
        (
            "timestamp",
            "screen_ppdeg_x_coordinate",
            "screen_ppdeg_y_coordinate",
            "eye1_pupil_size",#pa
            "eye2_pupil_size",#pa
        )
    )
)
bids_columns = []
for eyenum, eyename in enumerate(eye):
    for name in columns:
        colprefix = f"eye{eyenum + 1}" if name.endswith(f"_{eyename}") else ""
        _newname = name.split("_")[0]
        _newname = re.sub(r"([xy])$", r"_\1_coordinate", _newname)
        _newname = re.sub(r"([xy])vel$", r"_\1_velocity", _newname)
        _newname = _newname.split("_", 1)
        _newname[0] = EDF2BIDS_COLUMNS[_newname[0]]
        _newname.insert(0, colprefix)
        bids_columns.append("_".join((_n for _n in _newname if _n)))

# Rename columns to be BIDS-compliant
recording = recording.rename(columns=dict(zip(columns, bids_columns)))

# Reorder columns to render nicely (tracking first, pupil size after)
columns = sorted(
    set(recording.columns.values).intersection(BIDS_COLUMNS_ORDER),
    key=lambda entry: BIDS_COLUMNS_ORDER.index(entry),
)
columns += [c for c in recording.columns.values if c not in columns]
recording = recording.reindex(columns=columns)

print(recording)
assert len(recording) == raw_recording_len

        eye1_x_coordinate  eye1_y_coordinate  eye1_pupil_size  \
0              442.600006         336.399994            847.0   
1              442.200012         336.799988            847.0   
2              441.799988         337.100006            847.0   
3              442.100006         336.000000            847.0   
4              443.000000         335.000000            847.0   
...                   ...                ...              ...   
425452         395.299988         294.600006            879.0   
425453         395.600006         294.500000            880.0   
425454         395.700012         293.299988            879.0   
425455         395.799988         291.700012            877.0   
425456         395.700012         290.200012            876.0   

        eye1_pupil_x_coordinate  eye1_pupil_y_coordinate  \
0                       -4219.0                  -5121.0   
1                       -4222.0                  -5118.0   
2                       -4226.0        

# 3 Parsing the calibration messages

In [21]:
print(calibration)

    timestamp  trial                                            trialid
0      624794     -1  !CAL \n>>>>>>> CALIBRATION (HV5,P-CR) FOR RIGH...
1      624794     -1                           !CAL Calibration points:
2      624794     -1                !CAL -36.8, -43.5         0,      0
3      624794     -1                !CAL -35.3, -59.8         0,  -2457
4      624794     -1                !CAL -36.9, -27.8         0,   2457
5      624794     -1                !CAL -60.2, -45.3     -3474,      0
6      624794     -1                 !CAL -7.3, -42.1      3474,      0
7      624794     -1  !CAL eye check box: (L,R,T,B)\n\t  -66    -2  ...
8      624794     -1  !CAL href cal range: (L,R,T,B)\n\t-5211  5211 ...
9      624794     -1  !CAL Cal coeff:(X=a+bx+cy+dxx+eyy,Y=f+gx+goaly...
10     624794     -1      !CAL Prenormalize: offx, offy = -36.815 -43.5
11     624794     -1         !CAL Gains: cx:88.187 lx:124.778 rx:51.882
12     624794     -1       !CAL Gains: cy:171.577 ty:160.046 by:

In [22]:
# Parse calibration metadata
metadata["CalibrationCount"] = 0
if not calibration.empty:
    warn("Calibration of more than one eye is not implemented")
    calibration.trialid = calibration.trialid.str.replace("!CAL", "")
    calibration.trialid = calibration.trialid.str.strip()

    metadata["CalibrationLog"] = list(
        zip(
            calibration.timestamp.values.astype(int),
            calibration.trialid.values,
        )
    )

    calibrations_msg = calibration.trialid.str.startswith(
        "VALIDATION"
    ) & calibration.trialid.str.contains("ERROR")
    metadata["CalibrationCount"] = calibrations_msg.sum()

    calibration_last = calibration.index[calibrations_msg][-1]
    try:
        meta_calib = re.match(
            r"VALIDATION (?P<ctype>[\w\d]+) (?P<eyeid>[RL]+) (?P<eye>RIGHT|LEFT) "
            r"(?P<result>\w+) ERROR (?P<avg>-?\d+\.\d+) avg\. (?P<max>-?\d+\.\d+) max\s+"
            r"OFFSET (?P<offsetdeg>-?\d+\.\d+) deg\. "
            r"(?P<offsetxpix>-?\d+\.\d+),(?P<offsetypix>-?\d+\.\d+) pix\.",
            calibration.loc[calibration_last, "trialid"].strip(),
        ).groupdict()

        metadata["CalibrationType"] = meta_calib["ctype"]
        metadata["AverageCalibrationError"] = [float(meta_calib["avg"])]
        metadata["MaximalCalibrationError"] = [float(meta_calib["max"])]
        metadata["CalibrationResultQuality"] = [meta_calib["result"]]
        metadata["CalibrationResultOffset"] = [
            float(meta_calib["offsetdeg"]),
            (float(meta_calib["offsetxpix"]), float(meta_calib["offsetypix"])),
        ]
        metadata["CalibrationResultOffsetUnits"] = ["deg", "pixels"]
    except AttributeError:
        warn("Calibration data found but unsuccessfully parsed for results")
        
        
print(calibration)

    timestamp  trial                                            trialid
0      624794     -1  >>>>>>> CALIBRATION (HV5,P-CR) FOR RIGHT: <<<<...
1      624794     -1                                Calibration points:
2      624794     -1                     -36.8, -43.5         0,      0
3      624794     -1                     -35.3, -59.8         0,  -2457
4      624794     -1                     -36.9, -27.8         0,   2457
5      624794     -1                     -60.2, -45.3     -3474,      0
6      624794     -1                      -7.3, -42.1      3474,      0
7      624794     -1  eye check box: (L,R,T,B)\n\t  -66    -2   -63 ...
8      624794     -1  href cal range: (L,R,T,B)\n\t-5211  5211 -3686...
9      624794     -1  Cal coeff:(X=a+bx+cy+dxx+eyy,Y=f+gx+goaly+ixx+...
10     624794     -1           Prenormalize: offx, offy = -36.815 -43.5
11     624794     -1              Gains: cx:88.187 lx:124.778 rx:51.882
12     624794     -1            Gains: cy:171.577 ty:160.046 by:

/var/folders/x4/yl1kbpks5sxc3345y3ttk6gm0000gn/T/ipykernel_94087/3407936680.py:4: UserWarning: Calibration of more than one eye is not implemented
  warn("Calibration of more than one eye is not implemented")


# 4 Parsing the events dataframe

In [23]:
# events[
#     events["type"] == "saccade"
# ]

In [24]:
# print(events)
print(recording)

# Process events: first generate empty columns
recording["eye1_fixation"] = 0
recording["eye1_saccade"] = 0
recording["eye1_blink"] = 0

# Add fixations
for _, fixation_event in events[
    events["type"] == "fixation"
].iterrows():
    recording.loc[
        (recording["timestamp"] >= fixation_event["start"])
        & (recording["timestamp"] <= fixation_event["end"]),
        "eye1_fixation",
    ] = 1

# Add saccades, and blinks, which are a sub-event of saccades
for _, saccade_event in events[
    events["type"] == "saccade"
].iterrows():
    recording.loc[
        (recording["timestamp"] >= saccade_event["start"])
        & (recording["timestamp"] <= saccade_event["end"]),
        "eye1_saccade",
    ] = 1

    if saccade_event["contains_blink"] == 1: #Note here some version is "blink", depends on the item name
        recording.loc[
            (recording["timestamp"] >= saccade_event["start"])
            & (recording["timestamp"] <= saccade_event["end"]),
            "eye1_blink",
        ] = 1

        eye1_x_coordinate  eye1_y_coordinate  eye1_pupil_size  \
0              442.600006         336.399994            847.0   
1              442.200012         336.799988            847.0   
2              441.799988         337.100006            847.0   
3              442.100006         336.000000            847.0   
4              443.000000         335.000000            847.0   
...                   ...                ...              ...   
425452         395.299988         294.600006            879.0   
425453         395.600006         294.500000            880.0   
425454         395.700012         293.299988            879.0   
425455         395.799988         291.700012            877.0   
425456         395.700012         290.200012            876.0   

        eye1_pupil_x_coordinate  eye1_pupil_y_coordinate  \
0                       -4219.0                  -5121.0   
1                       -4222.0                  -5118.0   
2                       -4226.0        

In [25]:
print(recording)

        eye1_x_coordinate  eye1_y_coordinate  eye1_pupil_size  \
0              442.600006         336.399994            847.0   
1              442.200012         336.799988            847.0   
2              441.799988         337.100006            847.0   
3              442.100006         336.000000            847.0   
4              443.000000         335.000000            847.0   
...                   ...                ...              ...   
425452         395.299988         294.600006            879.0   
425453         395.600006         294.500000            880.0   
425454         395.700012         293.299988            879.0   
425455         395.799988         291.700012            877.0   
425456         395.700012         290.200012            876.0   

        eye1_pupil_x_coordinate  eye1_pupil_y_coordinate  \
0                       -4219.0                  -5121.0   
1                       -4222.0                  -5118.0   
2                       -4226.0        

# 5 Write the data into BIDS structure

In [26]:
from copy import deepcopy

metadata['Columns'] = recording.columns.tolist()
print(metadata)
save_metadata = deepcopy(metadata)
# metadata.pop('CalibrationLog', None)
# print(metadata)

{'StopTime': None, 'StartTime': 910995, 'SamplingFrequency': 1000, 'EyeTrackingMethod': 'P-CR', 'RecordedEye': 'right', 'ScreenAOIDefinition': ['square', [0, 800, 0, 600]], 'PupilFitMethod': 'ellipse', 'PupilFitMethodNumberOfParameters': 5, 'PupilFitParameters': [(1.01, 4.0), (0.15, 0.05), (0.65, 0.65), (0.0, 0.0, 0.3)], 'ValidationPosition': [['eye1', [400, 300]], ['eye1', [400, 51]], ['eye1', [400, 549]], ['eye1', [48, 300]], ['eye1', [752, 300]], ['eye1', [400, 300]], ['eye1', [400, 51]], ['eye1', [400, 549]], ['eye1', [48, 300]], ['eye1', [752, 300]]], 'ValidationErrors': [('eye1', 0.53, (0.1, -13.9)), ('eye1', 0.39, (5.1, -8.9)), ('eye1', 5340198.51, (99999600.0, 99999451.0)), ('eye1', 2.68, (-71.3, -3.8)), ('eye1', 1.02, (-19.9, -18.1)), ('eye1', 0.5, (5.9, -12.0)), ('eye1', 0.41, (-10.0, -4.6)), ('eye1', 0.48, (-10.6, -7.5)), ('eye1', 1.74, (-45.4, 13.4)), ('eye1', 1.54, (-30.8, -28.2))], 'PupilThreshold': [84], 'CornealReflectionThreshold': [208], 'LoggedMessages': [(703078, 'N

In [27]:
metadata = save_metadata

In [28]:

def convert_to_int(metadata):
    if 'CalibrationCount' in metadata:
        metadata['CalibrationCount'] = int(metadata['CalibrationCount']) if isinstance(metadata['CalibrationCount'], (np.int32, np.int64, int)) else metadata['CalibrationCount']
    if "CalibrationLog" in metadata:
        metadata["CalibrationLog"] = [(int(x[0]),x[1]) if isinstance(x[0], (np.int32, np.int64, int)) else x for x in metadata['CalibrationLog']]
    return metadata

        
convert_metadata = convert_to_int(metadata)
# print(convert_metadata)

In [30]:
import os
# Load the autoreload extension
%load_ext autoreload
# Set autoreload to update the modules every time before executing a new line of code
%autoreload 2

import importlib
from write_bids_yiwei import write_bids_from_df
out_dir = Path("/Users/cag/Documents/Dataset/datasets/250423/edf_250423/")
edf_extension = 'EDF'
edf_name = edf_name
filename = os.path.splitext(edf_name)[0]
print(f'bid filename: {filename}')

write_bids_from_df(
    recording, convert_metadata,
    out_dir,
    filename,
)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
bid filename: 034455_fixation_dots_T1weighted_2025-04-24_16h12.45.920


('/Users/cag/Documents/Dataset/datasets/250423/edf_250423/034455_fixation_dots_T1weighted_2025-04-24_16h12.45.920.tsv.gz',
 '/Users/cag/Documents/Dataset/datasets/250423/edf_250423/034455_fixation_dots_T1weighted_2025-04-24_16h12.45.920.json')

Now the files are generated.
- EDF Path
    - \<filename\>.EDF
    - \<filename\>.tsv.gz

In [78]:
print(recording)

        eye1_x_coordinate  eye1_y_coordinate  eye1_pupil_size  \
0              497.600006         309.200012            926.0   
1              497.500000         308.600006            925.0   
2              497.399994         308.299988            925.0   
3              498.100006         308.799988            928.0   
4              498.799988         309.399994            932.0   
...                   ...                ...              ...   
721516                NaN                NaN              NaN   
721517                NaN                NaN              NaN   
721518                NaN                NaN              NaN   
721519                NaN                NaN              NaN   
721520                NaN                NaN              NaN   

        eye1_pupil_x_coordinate  eye1_pupil_y_coordinate  \
0                       -1649.0                  -5999.0   
1                       -1650.0                  -6002.0   
2                       -1650.0        